# Required Assignment 31 – Natural Language Processing

## Building and Evaluating NLP Applications Using Hugging Face Transformers

Complete all **TODO** sections.

This notebook follows the concepts demonstrated in the lecture videos:
- Hugging Face pipelines
- Tokenizers
- Transformer models
- Sequence classification
- Transformer architectures


## Part A – NLP Pipelines

In [11]:
#%pip install transformers
#%pip install torch
#%pip install ipywidgets
#%pip install tf-keras

In [12]:
import torch
import transformers
from transformers import pipeline

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Pipeline import successful")

PyTorch: 2.14.0+cpu
Transformers: 4.57.1
Pipeline import successful


In [13]:
from transformers import pipeline

classifier = pipeline(
    "sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english",
    framework="pt"
)
texts = [
    "The lecture was engaging and easy to follow.",
    "The notebook is confusing."
]
results = classifier(texts)
results

Device set to use cpu


[{'label': 'POSITIVE', 'score': 0.9996544122695923},
 {'label': 'NEGATIVE', 'score': 0.999387264251709}]

****Sentiment Analysis:****

The sentiment-analysis pipeline classifies whether the meaning or opinion in a text is positive or negative. In this example, the first sentence is classified as positive and the second sentence is classified as negative. This represents a text classification task and can be used to analyse reviews or customer feedback.

In [14]:
# fill-mask
# force to use pytorch instead of keras due to version incmpatibiltiy in my pc. 
from transformers import pipeline
unmasker = pipeline(
    "fill-mask",
    model="distilroberta-base",
    framework="pt"
)
fill_mask_result = unmasker(
    "This course will teach you all about <mask> models.",
    top_k=2
)
fill_mask_result

Some weights of the model checkpoint at distilroberta-base were not used when initializing RobertaForMaskedLM: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing RobertaForMaskedLM from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaForMaskedLM from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Device set to use cpu


[{'score': 0.1961975395679474,
  'token': 30412,
  'token_str': ' mathematical',
  'sequence': 'This course will teach you all about mathematical models.'},
 {'score': 0.04052716866135597,
  'token': 38163,
  'token_str': ' computational',
  'sequence': 'This course will teach you all about computational models.'}]

****Fill-Mask:**** 
The fill-mask pipeline predicts suitable words for a masked position using the surrounding context. In this example, the model predicted words such as "mathematical" and "computational". This represents masked language modelling and can be used for word prediction and contextual language understanding.

In [15]:
# Named Entity Recognition (NER)
ner = pipeline(
    "ner",
    model="dslim/bert-base-NER",
    aggregation_strategy="simple",
    framework="pt")

ner_result = ner(
    "My name is Christina and I work at Telsa in Brooklyn."
)
ner_result

Some weights of the model checkpoint at dslim/bert-base-NER were not used when initializing BertForTokenClassification: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForTokenClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForTokenClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Device set to use cpu


[{'entity_group': 'PER',
  'score': np.float32(0.9943942),
  'word': 'Christina',
  'start': 11,
  'end': 20},
 {'entity_group': 'ORG',
  'score': np.float32(0.9959883),
  'word': 'Telsa',
  'start': 35,
  'end': 40},
 {'entity_group': 'LOC',
  'score': np.float32(0.99733174),
  'word': 'Brooklyn',
  'start': 44,
  'end': 52}]

****Named Entity Recognition (NER):****

The NER pipeline identifies important entities in a sentence and classifies them into categories such as person, organisation and location. In this example, Christina is identified as a person, Telsa as an organisation and Brooklyn as a location. NER is useful for information extraction from text.

In [16]:
question_answerer = pipeline(
    "question-answering",
    model="distilbert-base-cased-distilled-squad",
    framework="pt"
)

qa_result = question_answerer(
    question="Where do I work?",
    context="My name is Christina and I work at Telsa in Brooklyn."
)

qa_result

Device set to use cpu


{'score': 0.628704752161866, 'start': 35, 'end': 40, 'answer': 'Telsa'}

****Question Answering:****

The question-answering pipeline finds an answer from the given context. In this example, the model answers "Telsa" to the question "Where do I work?". This represents extractive question answering, where the answer is taken directly from the provided text.

In [17]:
# text-generation
generator = pipeline(
    "text-generation",
    model="distilgpt2",
    framework="pt"
)
generation_result = generator(
    "In this course, we will teach you how to",
    max_new_tokens=30
)
generation_result

Device set to use cpu
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


[{'generated_text': 'In this course, we will teach you how to use this technology to help you build a business. We will teach you how to use this technology to help you build a business. We will teach you'}]

****Text Generation:****
The text-generation pipeline continues a given prompt by predicting and generating the next tokens. In this example, the model extended the prompt into a longer sentence. This represents generative NLP and can be used for writing assistance, content generation and conversational applications.

## Part B – Tokenizer and Sequence Classification

In [18]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

checkpoint = "distilbert-base-uncased-finetuned-sst-2-english"

# TODO
tokenizer = AutoTokenizer.from_pretrained(checkpoint)
model = AutoModelForSequenceClassification.from_pretrained(checkpoint)

text = "This course is excellent."

inputs = tokenizer(text, return_tensors="pt")
print("Input IDs:")
print(inputs["input_ids"])

print("\nTokens:")
print(tokenizer.convert_ids_to_tokens(inputs["input_ids"][0]))

print("\nAttention Mask:")
print(inputs["attention_mask"])

print("\nDecoded Text:")
print(tokenizer.decode(inputs["input_ids"][0]))

print(inputs)

outputs = model(**inputs)
print(outputs.logits)

# TODO: Convert logits to probabilities
probabilities = torch.softmax(outputs.logits, dim=1)
probabilities


Input IDs:
tensor([[ 101, 2023, 2607, 2003, 6581, 1012,  102]])

Tokens:
['[CLS]', 'this', 'course', 'is', 'excellent', '.', '[SEP]']

Attention Mask:
tensor([[1, 1, 1, 1, 1, 1, 1]])

Decoded Text:
[CLS] this course is excellent. [SEP]
{'input_ids': tensor([[ 101, 2023, 2607, 2003, 6581, 1012,  102]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1]])}
tensor([[-4.2635,  4.6273]], grad_fn=<AddmmBackward0>)


tensor([[1.3764e-04, 9.9986e-01]], grad_fn=<SoftmaxBackward0>)

In [19]:
predicted_class_id = torch.argmax(probabilities, dim=1).item()
predicted_label = model.config.id2label[predicted_class_id]

print("Predicted Label:", predicted_label)

Predicted Label: POSITIVE


In [20]:
from transformers import AutoModel
base_model = AutoModel.from_pretrained(checkpoint)
base_outputs = base_model(**inputs)

print("Hidden state shape:")
print(base_outputs.last_hidden_state.shape)

Hidden state shape:
torch.Size([1, 7, 768])


## Reflection

Answer:
1. Why is tokenization necessary?
   It is necessary because Transformer models cannot process raw text directly.
   The tokenizer breaks the text into smaller tokens and converts them into numerical IDs that the model can understand and process.

2. What are the purposes of `[CLS]` and `[SEP]`?
   [CLS] is placed at the beginning of the input sequence and used as a special token for classification tasks.  
   [SEP] is placed at the end of a sentence and can also be used to separate two sentences in a paired input.

3. What are logits?
   Logits are the raw output scores produced by the model before they are converted into probabilities.
   A higher logit means the model gives more preference to that class. 

5. Why is Softmax applied?
   It is applied to convert the logits into probabilities.
   This makes the output easier to interpret and allows us to identify the class with the highest probability. 


## Part C – Transformer Architectures

1. **Why is BERT considered an encoder model?**

   BERT is considered an encoder model because it is mainly designed to understand and represent the input text. It looks at the context of the words in the sentence and is suitable for tasks such as sentiment analysis, text classification and named entity recognition.

2. **Why is GPT considered a decoder model?**

   GPT is considered a decoder model because it generates text by predicting the next word or token based on the words that came before it. Therefore, it is mainly suitable for generative tasks such as text completion and response generation.

3. **Which architecture would you choose for the following tasks?**

   - **Sentiment analysis:** Encoder-only model such as BERT, because the task requires understanding and classifying the input text.
   - **Text generation:** Decoder-only model such as GPT, because the task requires generating new text one token at a time.
   - **Translation:** Encoder-decoder model such as T5, because the encoder understands the input sentence and the decoder generates the translated output.

4. **Explain the role of attention.**

   Attention helps the model focus on the words that are most relevant when understanding a sentence. The meaning of one word may depend on other words around it, including words that are far apart. Attention helps the model identify these relationships and understand the context of the sentence.

## Part D – Critical Evaluation (500 words)

Discuss:

- strengths of pretrained LLMs
- limitations
- hallucinations
- bias
- shallow understanding
- resource requirements
- when fine-tuning is preferred
- ethical considerations


Pretrained Large Language Models (LLMs) have become useful for many NLP tasks because they are trained on very large amounts of text before being applied to a specific problem. One of their main strengths is versatility. 

A pretrained model can perform tasks such as text classification, question answering, summarisation and text generation without building a completely new model for every task. LLMs can also learn from examples given in a prompt and may develop capabilities that were not directly programmed during training. This makes pretrained models useful when an organisation wants to test an NLP solution quickly before investing in a customised model.


However, pretrained LLMs also have several limitations. One important problem is hallucination. An LLM may generate an answer that sounds confident and reasonable but is actually incorrect. 
This can be risky when the output is used in situations where accuracy is important. 

Another limitation is shallow understanding. Although LLMs can produce very natural language, they do not understand the world in the same way humans do. Their responses are mainly based on patterns learned from the training data. They may also have difficulty with ambiguity, cultural context, sarcasm and humour.

Bias is another concern. Since LLMs learn from large datasets, they may learn and reproduce biases that already exist in the training data. This means the model may not treat all groups or languages equally. Fairness and inclusivity therefore need to be considered when evaluating an NLP application. The module also highlights the importance of making NLP systems transparent and accessible to speakers of different languages.

Resource requirements are another limitation. Large models may contain millions or billions of parameters and require significant computing resources to train and run. This can increase the cost and complexity of developing and deploying an LLM solution. Pretrained models are useful because organisations can reuse existing language knowledge instead of training a model from scratch.

Fine-tuning is preferred when a general pretrained model does not perform well enough for a particular task or dataset. During fine-tuning, the pretrained model is trained further using a smaller labelled dataset for the target task. This allows the model to keep the general language knowledge learned during pretraining while adapting to a more specific application. It usually requires much less task-specific data, time and resources compared with training a new model from scratch.

Ethical considerations should also be included when deploying LLMs. Organisations need to consider fairness, bias, transparency, interpretability and inclusivity when evaluating model outputs. The aim should not only be to build more powerful NLP systems, but also systems that are safer, more sustainable and suitable for real-world use. Therefore, pretrained LLMs provide many benefits, but their outputs and limitations still need to be carefully evaluated before they are used in practical applications.
